# Notebook 03: model training and evaluation

## 01. Notebook Purpose

This notebook trains and evaluates baseline classification models for binary prediction of machine failure. It uses the prepared train-test datasets generated in Notebook 02, where leakage-prone failure-mode columns and identifier columns were excluded and the selected feature engineering and preprocessing steps were applied. Because the target is strongly imbalanced, the notebook compares models using evaluation metrics beyond accuracy, including precision, recall, F1-score, ROC-AUC, and Average Precision.

## 02. Imports and Loading Data Sets

In [1]:
import pandas as pd
import numpy as np

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

In [3]:
X_train = pd.read_csv("../data/processed/X_train_prepared.csv")
X_test = pd.read_csv("../data/processed/X_test_prepared.csv")
y_train = pd.read_csv("../data/processed/y_train.csv")
y_test = pd.read_csv("../data/processed/y_test.csv")

## 03. Checking Prepared Data

In [4]:
# Inspecting shapes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (8000, 10)
X_test shape: (2000, 10)
y_train shape: (8000, 1)
y_test shape: (2000, 1)


In [5]:
#shape-cleaning step for scikit 
y_train = y_train.squeeze()
y_test = y_test.squeeze()

print(type(y_train))
print(type(y_test))

<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>


In [7]:
#Check class balance
print("Training target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))

Training target distribution:
Machine failure
0    0.966125
1    0.033875
Name: proportion, dtype: float64

Test target distribution:
Machine failure
0    0.966
1    0.034
Name: proportion, dtype: float64


In [8]:
X_train.head()

,num__Air temperature [K],num__Process temperature [K],num__Rotational speed [rpm],num__Torque [Nm],num__Tool wear [min],num__Temperature difference [K],num__ToolWear_x_Torque,cat__Type_H,cat__Type_L,cat__Type_M
0,0.998914,0.604282,-0.460607,0.718305,-0.843997,-1.100842,-0.621945,0.0,0.0,1.0
1,-1.505194,-1.153260,-0.775574,0.638456,0.382263,1.299658,0.646140,0.0,0.0,1.0
2,0.498092,1.077466,-1.007654,0.558607,0.460870,0.599512,0.689544,0.0,0.0,1.0
3,-0.553633,-0.139294,-0.709265,1.626586,-0.372359,0.899575,0.151247,0.0,1.0,0.0
4,-1.455112,-1.018064,1.070019,-1.128202,-0.906882,1.399679,-1.016909,0.0,1.0,0.0


In [9]:
X_train.dtypes.value_counts()

float64    10
Name: count, dtype: int64

In [10]:
print("Missing values in X_train:", X_train.isna().sum().sum())
print("Missing values in X_test:", X_test.isna().sum().sum())

Missing values in X_train: 0
Missing values in X_test: 0


## 04. Defining Baseline Models

I define a a small and credible baseline comparison
- DummyClassifier
- LogisticRegression
- RandomForestClassifier

In [13]:
models = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

## 05. Defining Cross-Validation Strategy

Since the target is imbalanced, I use stratified cross-validation.

In [14]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision"
}

## 06. Comparing Baseline Models with Cross-Validation

Because machine failure is a strongly imbalanced target, accuracy alone is not sufficient to assess model quality. Precision shows how reliable the failure predictions are, recall measures how many real failures are detected, and F1-score summarizes the trade-off between both. ROC-AUC and Average Precision evaluate how well the model separates failure from non-failure cases across thresholds, with Average Precision being especially informative for imbalanced classification.

In [15]:
cv_results = []

for model_name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )
    
    cv_results.append({
        "Model": model_name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1-score": scores["test_f1"].mean(),
        "ROC-AUC": scores["test_roc_auc"].mean(),
        "Average Precision": scores["test_average_precision"].mean()
    })

cv_results_df = pd.DataFrame(cv_results).sort_values(
    by="Average Precision",
    ascending=False
)

cv_results_df

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC,Average Precision
2,Random Forest,0.986875,0.951051,0.645859,0.769231,0.973119,0.844017
1,Logistic Regression,0.818875,0.134574,0.800808,0.230318,0.896973,0.432933
0,Dummy,0.966125,0.000000,0.000000,0.000000,0.500000,0.033875


The cross-validation results show that Random Forest is the strongest baseline model overall. It achieves the highest F1-score, ROC-AUC, and Average Precision, while also maintaining strong precision and solid recall. Logistic Regression reaches higher recall, meaning it identifies more of the failure cases, but this comes at the cost of much lower precision and a substantially lower F1-score. The Dummy classifier performs well only in terms of accuracy because it mostly predicts the majority class, which confirms that accuracy alone is not an appropriate metric for this strongly imbalanced problem.

## 07. Training the Best Baseline Model

Based on the cross-validation comparison, Random Forest is selected as the best baseline model. Although Logistic Regression achieves higher recall, Random Forest provides a much better overall balance between identifying failure cases and limiting false positives, as reflected in its higher F1-score, ROC-AUC, and Average Precision.

In [17]:
best_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

The selected Random Forest model is now trained on the full prepared training set and then evaluated on the held-out test set.

In [18]:
best_model.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


## 08. Making Predictions on the Test Set

In [19]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

## 09. Evaluating on the Test Set

To assess how well the selected baseline generalizes to unseen data, the model is evaluated on the held-out test set using the same metrics considered during cross-validation.

In [20]:
test_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1-score": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_proba),
    "Average Precision": average_precision_score(y_test, y_proba)
}

test_metrics_df = pd.DataFrame([test_metrics])
test_metrics_df

,Accuracy,Precision,Recall,F1-score,ROC-AUC,Average Precision
0,0.988,0.958333,0.676471,0.793103,0.971167,0.8634


## 10. Detailed Error Breakdown

The classification report and confusion matrix provide a more detailed view of the selected model’s performance on the test set, especially with respect to the minority failure class.

In [21]:
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

Classification Report:

              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1932
           1       0.96      0.68      0.79        68

    accuracy                           0.99      2000
   macro avg       0.97      0.84      0.89      2000
weighted avg       0.99      0.99      0.99      2000



In [22]:
print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

Confusion Matrix:

[[1930    2]
 [  22   46]]


The model is:
- very reliable when it predicts failure
precision for class 1 = 0.96
- reasonably good, but not excellent, at catching all failures
recall for class 1 = 0.68

## 11. Conclusion

The selected Random Forest baseline performs well on the test set and clearly outperforms the naive benchmark. It identifies machine failures with high precision and strong overall performance, but its recall shows that some failure cases are still missed. This makes the model a solid baseline for the project, while indicating that further improvement should focus on increasing failure detection without excessively increasing false positives.